# 🇧🇩 Any people bd voice clone — উন্নত বাংলা ভয়েস ক্লোনিং (Google Colab)

**রানটাইম:** `Runtime → Change runtime type → T4 GPU` (বা এর চেয়ে ভালো)।

**এই নোটবুক চারটি প্রজেক্ট বিশ্লেষণ করে তৈরি:**

| প্রজেক্ট | কী নেওয়া হয়েছে |
|---|---|
| **OmniVoice** | মূল ইঞ্জিন: ৬০০+ ভাষা (বাংলাসহ), zero-shot ক্লোনিং, দ্রুত ইনফারেন্স, LoRA ফাইন-টিউনিং |
| **Bangla-Voice-Cloning-System** | বাংলা টেক্সট নরমালাইজেশন (সংখ্যা/তারিখ/টাকা), সম্মতি (consent) নীতি। *(zip-এ `src/models` ফোল্ডার নেই, তাই ওই সিস্টেম নিজে চালানো যায় না)* |
| **VoiceAPI** | রেফারেন্স অডিও ৫–৩০ সেকেন্ড ধারণা, প্রোফাইল সংরক্ষণ |
| **voicebox** | বড় লেখাকে বাক্য-চাঙ্কে ভাগ করে জোড়া লাগানো, ভয়েস প্রোফাইল লাইব্রেরি |

**যা নতুন/আরও উন্নত (চারটির কোনোটিতে নেই):**
1. রেফারেন্স অডিও স্বয়ংক্রিয় ক্লিন-আপ: নয়েজ রিমুভ, লাউডনেস নরমালাইজ, সেরা ৩–১০ সেকেন্ড অংশ বাছাই, কোয়ালিটি রিপোর্ট
2. বাংলা Whisper দিয়ে রেফারেন্স ট্রান্সক্রিপ্ট (নিজে ঠিক করার সুযোগসহ)
3. **Best-of-N + ASR যাচাই:** প্রতি বাক্য একাধিকবার বানিয়ে সবচেয়ে কম ভুল (CER) উচ্চারণ বেছে নেয়
4. উন্নত বাংলা নরমালাইজার (দেশীয় সংখ্যা: লাখ/কোটি, ফোন নম্বর, তারিখ, সময়, টাকা, শতাংশ, ইউনিকোড ঠিক করা, কাস্টম লেক্সিকন)
5. Google Drive-এ স্থায়ী ভয়েস লাইব্রেরি
6. ব্যাচ জেনারেশন (প্রতি লাইনে একটি অডিও + ZIP)
7. **LoRA ফাইন-টিউনিং পাইপলাইন:** ১০–৬০ মিনিটের রেকর্ডিং থেকে স্পিকারের জন্য আলাদা মডেল
8. বাংলা Gradio UI

> ⚠️ **দায়িত্বশীল ব্যবহার:** শুধু নিজের কণ্ঠ বা যার স্পষ্ট অনুমতি আছে তার কণ্ঠ ক্লোন করুন। প্রতারণা, ভুয়া খবর বা ছদ্মবেশ ধারণে ব্যবহার নিষিদ্ধ।

## ১) ইনস্টল ও সেটআপ

In [ ]:
#@title ইনস্টল (দ্রুত — যা নেই শুধু তাই ইনস্টল + দ্রুত সমান্তরাল ডাউনলোড)
import subprocess, os, sys, importlib.util, platform
# স্থায়ী লোকাল ফোল্ডার (সার্ভার রিস্টার্টের পরেও থাকে); অন্য জায়গা চাইলে VOICE_LAB_DIR এনভায়রনমেন্ট ভ্যারিয়েবল দিন
BASE_DIR = os.environ.get("VOICE_LAB_DIR", os.path.expanduser("~/BanglaVoiceLab"))
os.makedirs(f"{BASE_DIR}/models", exist_ok=True); os.makedirs(f"{BASE_DIR}/bin", exist_ok=True)
os.environ["HF_HOME"] = f"{BASE_DIR}/models"   # মডেল একবারই নামে; পরেরবার এখান থেকেই ব্যবহার হয় (আবার ডাউনলোড নয়)
subprocess.run("pip -q install uv", shell=True)
subprocess.run("uv pip install --system -q hf_transfer", shell=True)   # মডেল ডাউনলোড বহুগুণ দ্রুত করে (সেকেন্ডের কাজ)
# মডেল ফাইল ব্যাকগ্রাউন্ডে নামতে শুরু করে (ইনস্টলের সাথে একসাথে), ফলে পরে লোডে অপেক্ষা কমে
_PREFETCH = """
import os
try:
    import hf_transfer
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
except Exception:
    pass
from huggingface_hub import snapshot_download as d
_SKIP = ["*.bin", "*.h5", "*.msgpack", "flax_model*", "tf_model*", "onnx/*"]   # Whisper-এর অপ্রয়োজনীয় বাড়তি ফরম্যাট (কয়েক GB) নামানো হবে না
for r, ign in (("k2-fsa/OmniVoice", None), ("eustlb/higgs-audio-v2-tokenizer", None), ("openai/whisper-large-v3-turbo", _SKIP)):
    try: d(r, max_workers=16, ignore_patterns=ign)
    except Exception: pass
"""
subprocess.Popen([sys.executable, "-c", _PREFETCH], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
# শুধু যে প্যাকেজ নেই সেগুলোই ইনস্টল (আগে থেকে থাকলে সঙ্গে সঙ্গে পার)
_NEED = [("omnivoice", ["omnivoice"]), ("noisereduce", ["noisereduce"]), ("pyloudnorm", ["pyloudnorm"]),
         ("jiwer", ["jiwer"]), ("gradio", ["gradio"]), ("soundfile", ["soundfile"]), ("librosa", ["librosa"]),
         ("peft", ["peft"]), ("fastapi", ["fastapi"]), ("uvicorn", ["uvicorn"]),
         ("python-multipart", ["multipart", "python_multipart"])]
_missing = [p for p, mods in _NEED if not any(importlib.util.find_spec(m) for m in mods)]
if _missing:
    _PKGS = " ".join(_missing)
    print("ইনস্টল হচ্ছে:", _PKGS)
    _r = subprocess.run(f"uv pip install --system -q {_PKGS}", shell=True)
    if _r.returncode != 0:  # uv ব্যর্থ হলে সাধারণ pip
        subprocess.run(f"pip -q install {_PKGS}", shell=True)
else:
    print("সব প্যাকেজ আগে থেকেই আছে ✅")
# cloudflared ব্যাকগ্রাউন্ডে নামানো হচ্ছে, যাতে পরে অপেক্ষা করতে না হয়
CF_BIN = f"{BASE_DIR}/bin/cloudflared"
_CF_ARCH = "arm64" if platform.machine().lower() in ("aarch64", "arm64") else "amd64"
if not os.path.exists(CF_BIN):
    subprocess.Popen(f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{_CF_ARCH} -O {CF_BIN}.part && chmod +x {CF_BIN}.part && mv {CF_BIN}.part {CF_BIN}", shell=True)
import torch
print("CPU মোডে চলছে (GPU লাগবে না) | CPU কোর:", os.cpu_count())

In [ ]:
#@title Google Drive মাউন্ট (ভয়েস প্রোফাইল স্থায়ীভাবে সেভ করতে)
USE_DRIVE = True  #@param {type:"boolean"}
import os
# CPU/সার্ভারে Google Drive লাগে না: স্থায়ী লোকাল ফোল্ডার ব্যবহার হয় (USE_DRIVE এখানে কোনো প্রভাব ফেলে না)
BASE_DIR = globals().get("BASE_DIR") or os.environ.get("VOICE_LAB_DIR", os.path.expanduser("~/BanglaVoiceLab"))
LIB_DIR = BASE_DIR
WORK_DIR = f"{LIB_DIR}/tmp"
for _d in ("voices", "outputs", "models", "uploads", "tmp"):
    os.makedirs(f"{LIB_DIR}/{_d}", exist_ok=True)
print("লাইব্রেরি:", LIB_DIR)


## ২) বাংলা টেক্সট নরমালাইজার

In [ ]:
#@title বাংলা টেক্সট নরমালাইজার
import re, unicodedata

_SMALL = {
    0: 'শূন্য',
    1: 'এক',
    2: 'দুই',
    3: 'তিন',
    4: 'চার',
    5: 'পাঁচ',
    6: 'ছয়',
    7: 'সাত',
    8: 'আট',
    9: 'নয়',
    10: 'দশ',
    11: 'এগারো',
    12: 'বারো',
    13: 'তেরো',
    14: 'চৌদ্দ',
    15: 'পনেরো',
    16: 'ষোলো',
    17: 'সতেরো',
    18: 'আঠারো',
    19: 'উনিশ',
    20: 'বিশ',
    21: 'একুশ',
    22: 'বাইশ',
    23: 'তেইশ',
    24: 'চব্বিশ',
    25: 'পঁচিশ',
    26: 'ছাব্বিশ',
    27: 'সাতাশ',
    28: 'আঠাশ',
    29: 'ঊনত্রিশ',
    30: 'ত্রিশ',
    31: 'একত্রিশ',
    32: 'বত্রিশ',
    33: 'তেত্রিশ',
    34: 'চৌত্রিশ',
    35: 'পঁয়ত্রিশ',
    36: 'ছত্রিশ',
    37: 'সাঁইত্রিশ',
    38: 'আটত্রিশ',
    39: 'ঊনচল্লিশ',
    40: 'চল্লিশ',
    41: 'একচল্লিশ',
    42: 'বিয়াল্লিশ',
    43: 'তেতাল্লিশ',
    44: 'চুয়াল্লিশ',
    45: 'পঁয়তাল্লিশ',
    46: 'ছেচল্লিশ',
    47: 'সাতচল্লিশ',
    48: 'আটচল্লিশ',
    49: 'ঊনপঞ্চাশ',
    50: 'পঞ্চাশ',
    51: 'একান্ন',
    52: 'বাহান্ন',
    53: 'তিপ্পান্ন',
    54: 'চুয়ান্ন',
    55: 'পঞ্চান্ন',
    56: 'ছাপ্পান্ন',
    57: 'সাতান্ন',
    58: 'আটান্ন',
    59: 'ঊনষাট',
    60: 'ষাট',
    61: 'একষট্টি',
    62: 'বাষট্টি',
    63: 'তেষট্টি',
    64: 'চৌষট্টি',
    65: 'পঁয়ষট্টি',
    66: 'ছেষট্টি',
    67: 'সাতষট্টি',
    68: 'আটষট্টি',
    69: 'ঊনসত্তর',
    70: 'সত্তর',
    71: 'একাত্তর',
    72: 'বাহাত্তর',
    73: 'তিয়াত্তর',
    74: 'চুয়াত্তর',
    75: 'পঁচাত্তর',
    76: 'ছিয়াত্তর',
    77: 'সাতাত্তর',
    78: 'আটাত্তর',
    79: 'ঊনআশি',
    80: 'আশি',
    81: 'একাশি',
    82: 'বিরাশি',
    83: 'তিরাশি',
    84: 'চুরাশি',
    85: 'পঁচাশি',
    86: 'ছিয়াশি',
    87: 'সাতাশি',
    88: 'আটাশি',
    89: 'ঊননব্বই',
    90: 'নব্বই',
    91: 'একানব্বই',
    92: 'বিরানব্বই',
    93: 'তিরানব্বই',
    94: 'চুরানব্বই',
    95: 'পঁচানব্বই',
    96: 'ছিয়ানব্বই',
    97: 'সাতানব্বই',
    98: 'আটানব্বই',
    99: 'নিরানব্বই'
}

_BN2EN = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")
_MONTHS = ["", "জানুয়ারি", "ফেব্রুয়ারি", "মার্চ", "এপ্রিল", "মে", "জুন",
           "জুলাই", "আগস্ট", "সেপ্টেম্বর", "অক্টোবর", "নভেম্বর", "ডিসেম্বর"]

ABBREVIATIONS = {
    "ডা.": "ডাক্তার", "ড.": "ডক্টর", "মো.": "মোহাম্মদ", "মোঃ": "মোহাম্মদ", "মি.": "মিস্টার",
    "প্রফ.": "প্রফেসর", "নং": "নম্বর", "সে.মি.": "সেন্টিমিটার", "কি.মি.": "কিলোমিটার",
    "কে.জি.": "কিলোগ্রাম", "মি.লি.": "মিলিলিটার",
    "বি.এ.": "বি এ", "এম.এ.": "এম এ", "পিএইচডি": "পি এইচ ডি",
}

# নিজের শব্দ যোগ করুন: ইংরেজি/ব্র্যান্ড নাম কীভাবে পড়বে
CUSTOM_LEXICON = {
    "AI": "এআই", "Google": "গুগল", "YouTube": "ইউটিউব", "Facebook": "ফেসবুক",
    "WhatsApp": "হোয়াটসঅ্যাপ", "OK": "ওকে", "Mr.": "মিস্টার", "Dr.": "ডক্টর",
}

def int_to_bn_words(n: int) -> str:
    """বাংলা (দক্ষিণ এশীয়) পদ্ধতিতে: হাজার, লাখ, কোটি।"""
    if n < 0:
        return "ঋণাত্মক " + int_to_bn_words(-n)
    if n < 100:
        return _SMALL[n]
    parts = []
    for value, name in ((10_000_000, "কোটি"), (100_000, "লাখ"), (1000, "হাজার")):
        if n >= value:
            c, n = divmod(n, value)
            parts.append(int_to_bn_words(c) + " " + name)
    if n >= 100:
        h, n = divmod(n, 100)
        parts.append(_SMALL[h] + "শো")
    if n > 0:
        parts.append(_SMALL[n])
    return " ".join(parts) if parts else _SMALL[0]

def _num_str_to_words(s: str) -> str:
    s = s.replace(",", "")
    if "." in s:
        a, b = s.split(".", 1)
        out = int_to_bn_words(int(a or 0))
        if b:
            out += " দশমিক " + " ".join(_SMALL[int(d)] for d in b)
        return out
    return int_to_bn_words(int(s))

def _digits_spoken(s: str) -> str:
    return " ".join(_SMALL[int(d)] for d in s if d.isdigit())

class BanglaNormalizer:
    def __init__(self, lexicon=None, add_final_danda=True):
        self.lex = {**CUSTOM_LEXICON, **(lexicon or {})}
        self.add_final_danda = add_final_danda

    def _fix_unicode(self, t):
        t = t.replace("\ufeff", "").replace("\u200c", "").replace("\u200d", "")
        t = unicodedata.normalize("NFC", t)
        # NFC ড়/ঢ়/য় কে ভেঙে ফেলে, আবার জোড়া লাগাই (প্রচলিত বানান)
        t = t.replace("\u09a1\u09bc", "\u09dc").replace("\u09a2\u09bc", "\u09dd").replace("\u09af\u09bc", "\u09df")
        # ো / ৌ এর ভাঙা রূপ ঠিক করা
        t = t.replace("\u09c7\u09be", "\u09cb").replace("\u09c7\u09d7", "\u09cc")
        return t

    def _punct(self, t):
        for a, b in {"|": "।", "“": '"', "”": '"', "‘": "'", "’": "'", "…": "...", "—": ", ", "–": ", ",
                     "\u2018": "'", "\u2019": "'"}.items():
            t = t.replace(a, b)
        t = re.sub(r"[ \t]+", " ", t)
        t = re.sub(r"([।?!,;])\1+", r"\1", t)
        return t.strip()

    def normalize(self, text: str) -> str:
        t = self._fix_unicode(text)
        for k in sorted(ABBREVIATIONS, key=len, reverse=True):
            t = t.replace(k, ABBREVIATIONS[k])
        for k in sorted(self.lex, key=len, reverse=True):
            t = re.sub(r"(?<![A-Za-z])" + re.escape(k) + r"(?![A-Za-z])", self.lex[k], t)
        t = t.translate(_BN2EN)

        # তারিখ  ১৫/০৮/২০২৪ (দিন/মাস/বছর)
        def _date(m):
            d, mo, y = int(m.group(1)), int(m.group(2)), m.group(3)
            if 1 <= mo <= 12 and 1 <= d <= 31:
                return f"{int_to_bn_words(d)} {_MONTHS[mo]} {_num_str_to_words(y)}"
            return m.group(0)
        t = re.sub(r"\b(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})\b", _date, t)

        # সময় ৫:৩০
        def _time(m):
            h, mi = int(m.group(1)), int(m.group(2))
            if h < 24 and mi < 60:
                return f"{int_to_bn_words(h)}টা" + (f" {int_to_bn_words(mi)} মিনিট" if mi else "")
            return m.group(0)
        t = re.sub(r"\b(\d{1,2}):(\d{2})\b", _time, t)

        # ফোন নম্বর / লম্বা সংখ্যা: অঙ্ক ধরে ধরে
        t = re.sub(r"(?<![\d.,])\+?(?:88)?0?\d{9,}(?![\d])", lambda m: _digits_spoken(m.group(0)), t)

        # টাকা, ডলার, শতাংশ
        t = re.sub(r"৳\s*(\d[\d,]*(?:\.\d+)?)", lambda m: _num_str_to_words(m.group(1)) + " টাকা", t)
        t = re.sub(r"(\d[\d,]*(?:\.\d+)?)\s*৳", lambda m: _num_str_to_words(m.group(1)) + " টাকা", t)
        t = re.sub(r"\$\s*(\d[\d,]*(?:\.\d+)?)", lambda m: _num_str_to_words(m.group(1)) + " ডলার", t)
        t = re.sub(r"(\d[\d,]*(?:\.\d+)?)\s*%", lambda m: _num_str_to_words(m.group(1)) + " শতাংশ", t)

        # বাকি সংখ্যা
        t = re.sub(r"\d[\d,]*(?:\.\d+)?", lambda m: _num_str_to_words(m.group(0).rstrip(",")), t)

        t = self._punct(t)
        if self.add_final_danda and t and t[-1] not in "।?!.\"'":
            t += "।"
        return t

normalizer = BanglaNormalizer()
for s in ["আমার ২৫ টাকা আছে", "দাম ৳১,২৫,০০০ মাত্র", "তারিখ ১৫/০৮/২০২৪, সময় ৫:৩০", "ফোন: ০১৭১২৩৪৫৬৭৮", "৩.৫% বৃদ্ধি, ১২৩৪৫৬৭ জন"]:
    print(s, "→", normalizer.normalize(s))

## ৩) রেফারেন্স অডিও প্রসেসিং (ক্লিন-আপ + সেরা অংশ বাছাই)

In [ ]:
#@title রেফারেন্স অডিও টুলস
import os
import numpy as np, librosa, soundfile as sf, noisereduce as nr, pyloudnorm as pyln

SR = 24000

def load_mono(path, sr=SR):
    y, _ = librosa.load(path, sr=sr, mono=True)
    return y.astype(np.float32)

def enhance(y, sr=SR, denoise=True, target_lufs=-23.0):
    if denoise:
        y = nr.reduce_noise(y=y, sr=sr, stationary=False, prop_decrease=0.6)
    try:
        meter = pyln.Meter(sr)
        loud = meter.integrated_loudness(y)
        if np.isfinite(loud):
            y = pyln.normalize.loudness(y, loud, target_lufs)
    except Exception:
        pass
    peak = np.max(np.abs(y)) + 1e-9
    if peak > 0.97:
        y = y * (0.97 / peak)
    return y.astype(np.float32)

def best_segment(y, sr=SR, min_sec=4.0, max_sec=10.0, top_db=32):
    """কথা-সমৃদ্ধ, কম-নীরবতার সবচেয়ে ভালো ৪–১০ সেকেন্ড অংশ বেছে নেয়।"""
    segs = librosa.effects.split(y, top_db=top_db, frame_length=1024, hop_length=256)
    if len(segs) == 0:
        return y
    best, best_score = None, -1
    for i in range(len(segs)):
        start = segs[i][0]; speech = 0; end = start
        for j in range(i, len(segs)):
            gap = (segs[j][0] - end) / sr if j > i else 0
            span = (segs[j][1] - start) / sr
            if span > max_sec or gap > 0.8:
                break
            speech += (segs[j][1] - segs[j][0]) / sr
            end = segs[j][1]
            if speech >= min_sec:
                score = min(speech, 8.0) + 0.5 * (speech / max(span, 1e-3))
                if score > best_score:
                    best, best_score = (start, end), score
    if best is None:  # বিরতিহীন লম্বা বা খুব ছোট অডিও: max_sec পর্যন্ত নেওয়া
        seg = y[segs[0][0]: segs[-1][1]]
        return seg[: int(max_sec * sr)]
    pad = int(0.1 * sr)
    return y[max(0, best[0] - pad): min(len(y), best[1] + pad)]

def quality_report(y, sr=SR):
    dur = len(y) / sr
    frame = 1024
    rms = librosa.feature.rms(y=y, frame_length=frame, hop_length=256)[0] + 1e-9
    db = 20 * np.log10(rms)
    noise = np.percentile(db, 10); speech = np.percentile(db, 90)
    snr = float(speech - noise)
    clip = float(np.mean(np.abs(y) > 0.99))
    notes = []
    if dur < 3: notes.append("⚠️ অডিও ৩ সেকেন্ডের কম: আরও লম্বা নমুনা দিন")
    if snr < 15: notes.append("⚠️ ব্যাকগ্রাউন্ড নয়েজ বেশি (SNR কম): শান্ত ঘরে রেকর্ড করুন")
    if clip > 0.001: notes.append("⚠️ ক্লিপিং/ফাটা আওয়াজ আছে: মাইক থেকে একটু দূরে বা নিচু ভলিউমে রেকর্ড করুন")
    if not notes: notes.append("✅ ভালো মানের রেফারেন্স")
    return {"duration_sec": round(dur, 2), "snr_db": round(snr, 1), "clip_ratio": round(clip, 5), "notes": notes}

def prepare_reference(path, denoise=True, pick_best=True, out_path=None):
    out_path = out_path or f"{WORK_DIR}/ref_clean.wav"
    if not path or not os.path.isfile(path):
        raise FileNotFoundError(f"রেফারেন্স অডিও ফাইল পাওয়া যায়নি: {path}")
    try:
        y = load_mono(path)
    except Exception as e:
        raise ValueError(f"রেফারেন্স অডিও পড়া যায়নি (wav/mp3/flac/m4a দিন): {e}") from e
    if len(y) < SR * 1.0 or float(np.max(np.abs(y))) < 1e-4:
        raise ValueError("রেফারেন্স অডিও খুব ছোট বা ফাঁকা। ৫–১৫ সেকেন্ডের পরিষ্কার কথা দিন।")
    if pick_best:
        y = best_segment(y)          # আগে ছোট করা: নয়েজ রিমুভ অনেক দ্রুত হয়
    y = enhance(y, denoise=denoise)
    sf.write(out_path, y, SR)
    return out_path, quality_report(y)


## ৪) মডেল লোড (OmniVoice + বাংলা Whisper ASR)

In [ ]:
#@title মডেল লোড
import torch, time, threading, os
from omnivoice import OmniVoice, VoiceClonePrompt

MODEL_ID = "k2-fsa/OmniVoice"
torch.set_num_threads(os.cpu_count() or 1)
if globals().get("model") is None:   # একই মডেল দুইবার RAM-এ লোড হবে না
    try:
        model = OmniVoice.from_pretrained(MODEL_ID, device_map="cpu", dtype=torch.float32, load_asr=False)   # Whisper আলাদা থ্রেডে লোড হবে, অপেক্ষা করতে হবে না
    except Exception as e:
        raise RuntimeError(f"❌ মডেল ডাউনলোড/লোড ব্যর্থ ({MODEL_ID}): ইন্টারনেট ও ডিস্কের জায়গা দেখে সেলটি আবার চালান। বিস্তারিত: {e}") from e
# (CUDA/TF32/cudnn সেটিং CPU সার্ভারে লাগে না, তাই বন্ধ)
# if torch.cuda.is_available(): torch.backends.cuda.matmul.allow_tf32 = True
# if torch.cuda.is_available(): torch.backends.cudnn.benchmark = False  # দৈর্ঘ্য বদলালে বারবার টিউনিং করে ধীর করত
print("মডেল লোড হয়েছে ✅  | বাংলা (bn) সাপোর্টেড:", "bn" in model.supported_language_ids())

_asr_lock = threading.Lock()
def ensure_asr():
    """Whisper দরকার হলে (বা ব্যাকগ্রাউন্ডে) একবারই লোড হয়।"""
    with _asr_lock:
        if getattr(model, "_asr_pipe", None) is None:
            model.load_asr_model()
def _bg_asr():
    try: ensure_asr()
    except Exception as e: print("ASR ব্যাকগ্রাউন্ড লোড হয়নি:", e)
threading.Thread(target=_bg_asr, daemon=True).start()

def transcribe_bn(wav_path):
    """বাংলা Whisper ট্রান্সক্রিপশন (ভাষা জোর করে bengali ধরা হয়)।"""
    ensure_asr()
    y = load_mono(wav_path, 16000)
    try:
        out = model._asr_pipe({"raw": y, "sampling_rate": 16000},
                              generate_kwargs={"language": "bengali", "task": "transcribe"})
        return out["text"].strip()
    except Exception as e:
        print("fallback ASR:", e)
        return model.transcribe(wav_path)
# ওয়ার্ম-আপ এখানে আলাদাভাবে চালানো হয় না (সময় বাঁচাতে) — Turbo Engine সেলের ব্যাচ ওয়ার্ম-আপই কার্নেল গরম করে দেয়


## ৫) ভয়েস লাইব্রেরি (সেভ / লোড)

In [ ]:
#@title ভয়েস লাইব্রেরি
import json, os, shutil, datetime, re

def _slug(name):
    return re.sub(r"[^\w\u0980-\u09FF\-]+", "_", name.strip()) or "voice"

def create_voice(name, audio_path, ref_text=None, consent=False, denoise=True, pick_best=True):
    if not consent:
        raise ValueError("সম্মতি (consent) নিশ্চিত করুন: এই কণ্ঠ আপনার নিজের বা অনুমতিপ্রাপ্ত।")
    vid = _slug(name)
    vdir = f"{LIB_DIR}/voices/{vid}"
    os.makedirs(vdir, exist_ok=True)
    clean, report = prepare_reference(audio_path, denoise=denoise, pick_best=pick_best, out_path=f"{vdir}/ref.wav")
    text = (ref_text or "").strip() or transcribe_bn(clean)
    text = normalizer.normalize(text)
    prompt = model.create_voice_clone_prompt(ref_audio=clean, ref_text=text)
    prompt.save(f"{vdir}/prompt.pt")
    meta = {"name": name, "ref_text": text, "created": datetime.datetime.now().isoformat(timespec="seconds"),
            "consent_confirmed": True, "quality": report, "model": MODEL_ID}
    json.dump(meta, open(f"{vdir}/meta.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    return vid, meta

def list_voices():
    d = f"{LIB_DIR}/voices"
    return sorted(x for x in os.listdir(d) if os.path.exists(f"{d}/{x}/prompt.pt")) if os.path.isdir(d) else []

_cache = {}
def load_voice(vid):
    if vid not in _cache:
        _cache[vid] = VoiceClonePrompt.load(f"{LIB_DIR}/voices/{vid}/prompt.pt")
    return _cache[vid]

def delete_voice(vid):
    shutil.rmtree(f"{LIB_DIR}/voices/{vid}", ignore_errors=True); _cache.pop(vid, None)

print("সংরক্ষিত ভয়েস:", list_voices())

## ৬) সিনথেসিস ইঞ্জিন (চাঙ্কিং + Best-of-N + ASR যাচাই)

In [ ]:
#@title সিনথেসিস ইঞ্জিন
import numpy as np, jiwer, re, time, random, torch

_SENT_SPLIT = re.compile(r"(?<=[।?!])\s*|\n+")
_SOFT_SPLIT = re.compile(r"(?<=[,;:،])\s+")

def split_text(text, max_chars=160):
    """বাক্য → চাঙ্ক। প্রতিটির সাথে পরবর্তী বিরতির (pause) সময় (সেকেন্ড)।"""
    items = []
    for para in re.split(r"\n\s*\n", text.strip()):
        sents = [s.strip() for s in _SENT_SPLIT.split(para) if s.strip()]
        for si, s in enumerate(sents):
            pieces = [s]
            if len(s) > max_chars:
                parts, cur = _SOFT_SPLIT.split(s), ""
                pieces = []
                for p in parts:
                    if cur and len(cur) + len(p) + 1 > max_chars:
                        pieces.append(cur); cur = p
                    else:
                        cur = (cur + " " + p).strip()
                if cur: pieces.append(cur)
                final = []
                for p in pieces:  # এখনও লম্বা হলে শব্দের সীমানায় কাটা
                    while len(p) > max_chars * 1.4:
                        cut = p.rfind(" ", 0, max_chars)
                        cut = cut if cut > 20 else max_chars
                        final.append(p[:cut]); p = p[cut:].strip()
                    final.append(p)
                pieces = final
            for pi, p in enumerate(pieces):
                last_piece = pi == len(pieces) - 1
                if not last_piece:
                    pause = 0.15
                elif si == len(sents) - 1:
                    pause = 0.7  # অনুচ্ছেদ শেষ
                elif p[-1] in "?!":
                    pause = 0.45
                else:
                    pause = 0.35
                items.append((p, pause))
    return items

def _clean_for_cer(s):
    return re.sub(r"[^\u0980-\u09FFA-Za-z0-9]", "", s)

def _fade(y, sr, ms=12):
    n = int(sr * ms / 1000)
    if len(y) > 2 * n:
        y = y.copy(); y[:n] *= np.linspace(0, 1, n); y[-n:] *= np.linspace(1, 0, n)
    return y

def synthesize(text, voice_id=None, speed=1.0, num_step=16, guidance_scale=2.0, seed=None,
               best_of=1, do_normalize=True, max_chars=160, instruct=None, prompt=None, progress=None):
    """রিটার্ন: (audio float32 @24k, report string)"""
    t0 = time.time()
    if do_normalize:
        text = "\n".join(normalizer.normalize(line) if line.strip() else "" for line in text.split("\n"))
    prompt = prompt or (load_voice(voice_id) if voice_id else None)
    items = split_text(text, max_chars)
    if not items:
        raise ValueError("টেক্সট খালি")
    base_seed = seed if seed not in (None, -1) else random.randint(0, 10**6)
    if best_of > 1: ensure_asr()
    use_asr = best_of > 1 and getattr(model, "_asr_pipe", None) is not None
    out, log = [], []
    for i, (chunk, pause) in enumerate(items):
        if progress: progress(i / len(items), desc=f"{int(i / len(items) * 100)}% — চাঙ্ক {i+1}/{len(items)}")
        best_audio, best_cer = None, 9.9
        for k in range(max(1, best_of)):
            torch.manual_seed(base_seed + i * 1000 + k)
            kw = dict(text=chunk, language="bn", speed=speed, num_step=num_step, guidance_scale=guidance_scale)
            if prompt is not None: kw["voice_clone_prompt"] = prompt
            if instruct: kw["instruct"] = instruct
            audio = model.generate(**kw)[0].astype(np.float32)
            if not use_asr:
                best_audio, best_cer = audio, -1
                break
            hyp = normalizer.normalize(transcribe_array(audio))
            c = jiwer.cer(_clean_for_cer(re.sub(r"\[[^\]]*\]", "", chunk)), _clean_for_cer(hyp) or "x")
            if c < best_cer:
                best_audio, best_cer = audio, c
            if c <= 0.08:  # যথেষ্ট ভালো, আর চেষ্টা লাগবে না
                break
        out.append(_fade(best_audio, SR)); out.append(np.zeros(int(SR * pause), dtype=np.float32))
        log.append(f"[{i+1}] CER={best_cer:.2f} | {chunk[:45]}" if best_cer >= 0 else f"[{i+1}] {chunk[:45]}")
    if progress: progress(1.0, desc="১০০% — সম্পন্ন")
    y = np.concatenate(out)
    y = y / (np.max(np.abs(y)) + 1e-9) * 0.89  # ≈ -1 dBFS
    dur = len(y) / SR
    rep = f"সময়: {dur:.1f}s | জেনারেশন: {time.time()-t0:.1f}s (RTF {(time.time()-t0)/max(dur,1e-3):.2f}) | চাঙ্ক: {len(items)} | seed: {base_seed}\n" + "\n".join(log)
    return y, rep

def transcribe_array(y, sr=SR):
    ensure_asr()
    y16 = librosa.resample(y, orig_sr=sr, target_sr=16000)
    return model._asr_pipe({"raw": y16, "sampling_rate": 16000},
                           generate_kwargs={"language": "bengali", "task": "transcribe"})["text"].strip()

def console_progress(prefix="জেনারেট"):
    """Colab আউটপুটে লাইভ প্রগ্রেস বার + শতাংশ দেখায়: synthesize(..., progress=console_progress())"""
    def cb(frac, desc=""):
        frac = max(0.0, min(1.0, float(frac)))
        n = int(frac * 20)
        line = f"{prefix} [{'█' * n}{'░' * (20 - n)}] {frac * 100:5.1f}%  {desc}"
        print("\r" + line.ljust(90), end="\n" if frac >= 1.0 else "", flush=True)
    return cb

print("প্রস্তুত | টেস্ট চাঙ্কিং:", split_text("আমি বাংলায় কথা বলি। তুমি কেমন আছো? ভালো আছি, ধন্যবাদ!"))

## ⚡ ৬ক) Turbo Engine (OmniVoice-এর সোর্স কোড পড়ে তৈরি)

OmniVoice-এর `generate()` কীভাবে কাজ করে তা পড়ে এখানে চারটি আসল স্পিডআপ বসানো হয়েছে — **আগের কোড অপরিবর্তিত**, শুধু এই নতুন ইঞ্জিন যোগ হয়েছে:

| কী | কেন দ্রুত |
|---|---|
| **KV-cache বন্ধ** | মডেল প্রতি ধাপে পুরো সিকোয়েন্স নতুন করে হিসাব করে; cache শুধু বাড়তি খরচ |
| **ব্যাচ জেনারেশন** | `generate(text=[...])` একাধিক বাক্য একসাথে GPU-তে চালায়; আগে একটি একটি করে চলত |
| **pydub সাইলেন্স-রিমুভ বন্ধ** | ডিফল্ট পোস্ট-প্রসেস CPU-তে ধীর; তার বদলে হালকা numpy ট্রিম |
| **প্রথম বাক্য আলাদা + কম স্টেপ** | প্রথম শব্দ দ্রুত আসে; বাকিগুলো বাজার সময় পেছনে তৈরি হয় |
| **LRU ক্যাশ** | একই বাক্য আবার বললে জেনারেশন ছাড়াই সঙ্গে সঙ্গে |

`/api/tts` (আপনার index.html যেটা ব্যবহার করে) এখন এই ইঞ্জিনেই চলে, আর `/live` পেজে রিয়েল-টাইম স্ট্রিমিং হয়।

In [ ]:
#@title ⚡ Turbo Engine — OmniVoice সোর্স কোড থেকে তৈরি (ব্যাচিং + স্ট্রিমিং + ক্যাশ)
import numpy as np, torch, time, re, random, threading, queue, collections, contextlib

# ── ১) KV-cache বন্ধ ────────────────────────────────────────────────
# OmniVoice প্রতি ধাপে পুরো সিকোয়েন্স bidirectional-ভাবে নতুন করে হিসাব করে, তাই HF-এর DynamicCache
# শুধু বাড়তি খরচ (omnivoice_flashinfer.py-র নিজের মন্তব্যেও এটাই লেখা আছে)।
try:
    model.llm.config.use_cache = False
except Exception as _e:
    print("use_cache বন্ধ করা যায়নি:", _e)

# ── ২) ছোট LRU ক্যাশ: একই বাক্য আবার বললে সঙ্গে সঙ্গে ─────────────────
_TC = collections.OrderedDict(); _TC_LOCK = threading.Lock(); _TC_MAX = 256
def _tc_get(k):
    with _TC_LOCK:
        v = _TC.get(k)
        if v is not None: _TC.move_to_end(k)
        return v
def _tc_put(k, v):
    with _TC_LOCK:
        _TC[k] = v; _TC.move_to_end(k)
        while len(_TC) > _TC_MAX: _TC.popitem(last=False)
def turbo_cache_clear():
    with _TC_LOCK: _TC.clear()

# ── ৩) হালকা পোস্ট-প্রসেস ───────────────────────────────────────────
# OmniVoice-এর ডিফল্ট পোস্ট-প্রসেস pydub দিয়ে CPU-তে সাইলেন্স কাটে (ধীর) এবং দুই পাশে ০.১s ফাঁকা যোগ করে।
# এখানে সেটি বন্ধ রেখে সস্তা numpy ট্রিম ব্যবহার হয়েছে; ভলিউম (ref_rms) মডেল নিজেই ঠিক করে দেয়।
def _trim_edges(y, sr=SR, keep_ms=70, rel_db=-45):
    if len(y) == 0: return y
    a = np.abs(y); pk = float(a.max())
    if pk < 1e-4: return y
    idx = np.flatnonzero(a > pk * (10 ** (rel_db / 20)))
    if len(idx) == 0: return y
    k = int(sr * keep_ms / 1000)
    return y[max(0, idx[0] - k): min(len(y), idx[-1] + k)]

def _gen_batch(texts, prompt, speed, steps, guide, seed):
    """একাধিক বাক্য একসাথে একটি ব্যাচে (public generate API; লিস্ট দিলে মডেল নিজেই ব্যাচ করে)।"""
    def run(tx):
        torch.manual_seed(seed)
        outs = model.generate(text=tx, language="bn", voice_clone_prompt=prompt, speed=speed,
                              num_step=steps, guidance_scale=guide,
                              postprocess_output=False, pad_duration=0.0, fade_duration=0.0)
        return [_fade(_trim_edges(np.asarray(o, dtype=np.float32).reshape(-1)), SR) for o in outs]
    try:
        return run(texts)
    except Exception as e:                      # ব্যাচ ব্যর্থ (যেমন VRAM) হলে একটি একটি করে
        print("ব্যাচ ব্যর্থ, একটি একটি করে চলছে:", str(e)[:120])
        import gc; gc.collect()
        return [run([t])[0] for t in texts]

# ── ৪) স্ট্রিমিং জেনারেটর ──────────────────────────────────────────
TURBO_LAST = {}
def turbo_stream(text, voice_id, speed=1.0, steps=12, first_steps=8, guide=2.0, seed=-1,
                 do_normalize=True, max_chars=160, batch_chars=240, lock=None, stats=None):
    """(audio float32 @24k, pause_sec) টুকরো টুকরো করে yield করে।
    • প্রথম ব্যাচ = শুধু প্রথম বাক্য, কম স্টেপে → প্রথম শব্দ খুব দ্রুত আসে।
    • বাকি বাক্য ~batch_chars অক্ষরের ব্যাচে একসাথে তৈরি হয় (GPU-তে ব্যাচ অনেক সস্তা)।
    • জেনারেশন আলাদা থ্রেডে এগিয়ে থাকে, বাজানো ও জেনারেশন একসাথে চলে।"""
    t0 = time.time()
    st = stats if stats is not None else {}
    if do_normalize:
        text = "\n".join(normalizer.normalize(l) if l.strip() else "" for l in text.split("\n"))
    prompt = load_voice(voice_id)
    items = split_text(text, max_chars)
    if not items: raise ValueError("টেক্সট খালি")
    if len(items[0][0]) > 90:                   # লম্বা প্রথম বাক্য কমা-তে ভেঙে দ্রুত শুরু
        c, p = items[0]; m = re.search(r"[,;:،]\s", c[20:90])
        if m:
            k = 20 + m.end(); items[0:1] = [(c[:k].strip(), 0.08), (c[k:].strip(), p)]
    batches, cur, cc = [], [], 0
    for it in items:
        if not batches and not cur: batches.append([it]); continue
        cur.append(it); cc += len(it[0])
        if cc >= batch_chars or len(cur) >= 6: batches.append(cur); cur, cc = [], 0
    if cur: batches.append(cur)
    base = seed if seed not in (None, -1) else random.randint(0, 10**6)
    fs = first_steps or steps
    q, stop = queue.Queue(), threading.Event()
    st.update(batches=len(batches), sentences=len(items), cache_hits=0, first_audio_s=None)

    def producer():
        try:
            with (lock if lock is not None else contextlib.nullcontext()):
                for bi, batch in enumerate(batches):
                    if stop.is_set(): break
                    stp = fs if bi == 0 else steps
                    keys = [(voice_id, t, round(speed, 3), stp, round(guide, 2)) for t, _ in batch]
                    aud = [_tc_get(k) for k in keys]
                    todo = [j for j, a in enumerate(aud) if a is None]
                    st["cache_hits"] += len(batch) - len(todo)
                    if todo:
                        outs = _gen_batch([batch[j][0] for j in todo], prompt, speed, stp, guide, base + bi * 1000)
                        for j, o in zip(todo, outs): aud[j] = o; _tc_put(keys[j], o)
                    for (_, pause), a in zip(batch, aud): q.put((a, pause))
        except Exception as e:
            q.put(e)
        finally:
            q.put(None)

    th = threading.Thread(target=producer, daemon=True); th.start()
    audio_sec = 0.0
    try:
        while True:
            it = q.get()
            if it is None: break
            if isinstance(it, Exception): raise it
            if st["first_audio_s"] is None: st["first_audio_s"] = round(time.time() - t0, 3)
            audio_sec += len(it[0]) / SR + it[1]
            yield it
    finally:
        stop.set()
        el = time.time() - t0
        st.update(total_s=round(el, 3), audio_s=round(audio_sec, 2), rtf=round(el / max(audio_sec, 1e-3), 3))
        TURBO_LAST.clear(); TURBO_LAST.update(st)

def synthesize_fast(text, voice_id=None, speed=1.0, num_step=16, guidance_scale=2.0, seed=None,
                    best_of=1, do_normalize=True, max_chars=160, instruct=None, prompt=None, progress=None):
    """synthesize()-র মতোই (audio, report) দেয়, কিন্তু ব্যাচিং/ক্যাশ সহ দ্রুত।
    best_of>1 / instruct / কাস্টম prompt হলে আগের ইঞ্জিনেই ফিরে যায়।"""
    slow = lambda: synthesize(text, voice_id, speed=speed, num_step=num_step, guidance_scale=guidance_scale,
                              seed=seed, best_of=best_of, do_normalize=do_normalize, max_chars=max_chars,
                              instruct=instruct, prompt=prompt, progress=progress)
    if best_of > 1 or instruct or prompt is not None or not voice_id: return slow()
    try:
        st, parts = {}, []
        if progress: progress(0.0, desc="০% — শুরু")
        for a, pause in turbo_stream(text, voice_id, speed=speed, steps=num_step, first_steps=num_step,
                                     guide=guidance_scale, seed=-1 if seed is None else seed,
                                     do_normalize=do_normalize, max_chars=max_chars, stats=st):
            parts += [a, np.zeros(int(SR * pause), dtype=np.float32)]
            if progress:
                _done, _tot = len(parts) // 2, max(1, st.get("sentences", 1))
                progress(min(_done / _tot, 1.0), desc=f"{int(min(_done / _tot, 1.0) * 100)}% — বাক্য {_done}/{_tot}")
        if progress: progress(1.0, desc="১০০% — সম্পন্ন")
        y = np.concatenate(parts); y = y / (np.max(np.abs(y)) + 1e-9) * 0.89
        rep = (f"⚡ Turbo | সময়: {len(y)/SR:.1f}s | জেনারেশন: {st['total_s']:.1f}s (RTF {st['rtf']:.2f}) | "
               f"প্রথম শব্দ: {st['first_audio_s']}s | ব্যাচ: {st['batches']} | ক্যাশ-হিট: {st['cache_hits']}")
        return y, rep
    except Exception as e:
        print("Turbo ব্যর্থ, আগের ইঞ্জিনে ফিরছি:", str(e)[:150]); return slow()

# ── ৫) ওয়ার্ম-আপ: ব্যাচ-আকারের কার্নেল আগে থেকেই গরম করা ───────────────
try:
    model.generate(text=["আমি বাংলায় কথা বলি।", "তুমি কেমন আছো?"], language="bn", num_step=4,
                   postprocess_output=False)
    print("⚡ Turbo Engine প্রস্তুত — ব্যাচ ওয়ার্ম-আপ সম্পন্ন")
except Exception as e:
    print("ওয়ার্ম-আপ বাদ:", str(e)[:120])

## ৭) দ্রুত পরীক্ষা (UI ছাড়া)
নিজের রেকর্ড করা ৫–১৫ সেকেন্ডের পরিষ্কার বাংলা অডিও আপলোড করুন।

In [ ]:
#@title ভয়েস তৈরি ও পরীক্ষা
import os
from IPython.display import Audio, display
import soundfile as sf

VOICE_NAME = "my_voice"  #@param {type:"string"}
I_HAVE_CONSENT = True  #@param {type:"boolean"}
TEST_TEXT = "আসসালামু আলাইকুম। আজ ২৫শে মার্চ, বাংলাদেশের ইতিহাসে একটি গুরুত্বপূর্ণ দিন।"  #@param {type:"string"}

REF_AUDIO_PATH = ""  #@param {type:"string"}
UPLOAD_DIR = f"{LIB_DIR}/uploads"; os.makedirs(UPLOAD_DIR, exist_ok=True)
_AUD = (".wav", ".mp3", ".flac", ".m4a", ".ogg", ".opus", ".webm")

def _find_reference():
    if REF_AUDIO_PATH.strip():
        p = os.path.expanduser(REF_AUDIO_PATH.strip())
        if not os.path.isfile(p): raise FileNotFoundError(f"রেফারেন্স অডিও পাওয়া যায়নি: {p}")
        return p
    fs = [os.path.join(UPLOAD_DIR, f) for f in os.listdir(UPLOAD_DIR) if f.lower().endswith(_AUD)]
    return max(fs, key=os.path.getmtime) if fs else None

def _show_uploader():
    try:
        import ipywidgets as w
    except Exception:
        print(f"অডিও ফাইলটি {UPLOAD_DIR} ফোল্ডারে রাখুন বা REF_AUDIO_PATH-এ পথ দিন, তারপর সেলটি আবার চালান।")
        return
    up = w.FileUpload(accept="audio/*", multiple=False)
    def _save(_):
        v = up.value
        for it in (list(v.values()) if isinstance(v, dict) else list(v)):
            name = os.path.basename(it["name"])
            with open(f"{UPLOAD_DIR}/{name}", "wb") as fo: fo.write(bytes(it["content"]))
            print("✅ আপলোড হয়েছে:", name, "— এখন এই সেলটি আবার চালান")
    up.observe(_save, names="value"); display(up)

ref_path = _find_reference()
if not ref_path:
    print("⚠️ রেফারেন্স অডিও নেই। নিচের বাটনে অডিও আপলোড করে সেলটি আবার চালান।")
    _show_uploader()
else:
  print("রেফারেন্স অডিও:", ref_path)
  vid, meta = create_voice(VOICE_NAME, ref_path, consent=I_HAVE_CONSENT)
  print(json.dumps(meta, ensure_ascii=False, indent=2))
  display(Audio(f"{LIB_DIR}/voices/{vid}/ref.wav"))

  y, rep = synthesize(TEST_TEXT, vid, best_of=1, progress=console_progress())
  print(rep)
  sf.write(f"{LIB_DIR}/outputs/test.wav", y, SR)
  display(Audio(y, rate=SR))

In [ ]:
#@title ⚡ গতি পরীক্ষা: আগের ইঞ্জিন বনাম Turbo (ভয়েস তৈরির পরে চালান)
# disabled on CPU (benchmark not needed)
# BENCH_TEXT = "আসসালামু আলাইকুম। আজ আবহাওয়া খুব সুন্দর। চলুন একসাথে ঘুরতে যাই। বিকেলে আমরা নদীর ধারে হাঁটব এবং চা খাব।"  #@param {type:"string"}
# _vs = list_voices()
# if not _vs:
#     print("আগে একটি ভয়েস তৈরি করুন (উপরের ভয়েস তৈরি সেল)")
# else:
#     _vid = _vs[0]
#     t = time.time(); y0, _ = synthesize(BENCH_TEXT, _vid, num_step=16, best_of=1); t_old = time.time() - t
#     turbo_cache_clear(); st = {}; t = time.time(); n = 0
#     for a, p in turbo_stream(BENCH_TEXT, _vid, steps=12, first_steps=8, stats=st): n += len(a)
#     t_new = time.time() - t
#     print(f"আগের ইঞ্জিন : সম্পূর্ণ অডিও পেতে {t_old:.1f}s")
#     print(f"Turbo        : প্রথম শব্দ {st['first_audio_s']}s, সম্পূর্ণ {t_new:.1f}s | RTF {st['rtf']} (১-এর কম = বলার চেয়ে দ্রুত)")
#     t = time.time(); [None for _ in turbo_stream(BENCH_TEXT, _vid, steps=12, first_steps=8)]
#     print(f"ক্যাশ থেকে আবার: {time.time()-t:.2f}s")

## ৭ক) 🌐 লাইভ URL + index.html — Any people bd voice clone

1. নিচের প্রথম সেল চালান — এটি `index_v2.html` নিজে থেকেই Drive-এ বানিয়ে দেবে (হাতে রাখতে হবে না)।
2. তারপর সার্ভার সেল চালান। এটি একটি **লাইভ URL** ও **টোকেন** দেবে।
3. ওই লিংক ব্রাউজারে খুললেই UI আসবে (সার্ভারই Drive থেকে index.html দেখায়)।
- Colab রানটাইম চালু থাকা পর্যন্ত লিংক কাজ করবে। বন্ধ হলে সেলটি আবার চালান।
- টোকেন ছাড়া কেউ আপনার GPU ব্যবহার করতে পারবে না।
- 🎧 **রিয়েল-টাইম শোনা:** সার্ভার চালু হলে যে `/live` লিংক দেখাবে সেটি খুলুন — প্রতিটি বাক্য তৈরি হওয়ামাত্র বাজতে থাকবে, পুরো অডিওর জন্য অপেক্ষা নেই।
- ⚡ `/api/tts` (index.html যেটা ব্যবহার করে) এখন Turbo Engine-এ চলে — index.html অপরিবর্তিত থাকলেও আগের চেয়ে দ্রুত।

In [ ]:
#@title index.html স্বয়ংক্রিয়ভাবে সেটআপ (Any people bd voice clone UI)
import base64, os
_INDEX_B64 = (
    "..."
    "Cjwvc2NyaXB0Pgo8L2JvZHk+CjwvaHRtbD4K"
)
os.makedirs(LIB_DIR, exist_ok=True)
with open(f"{LIB_DIR}/index_v2.html", "wb") as _f:
    _f.write(base64.b64decode(_INDEX_B64))
print("✅ index_v2.html তৈরি হয়েছে:", f"{LIB_DIR}/index_v2.html", "— Drive-এ হাতে রাখার আর দরকার নেই")


In [ ]:
#@title লাইভ সার্ভার চালু করুন (FastAPI + Cloudflare টানেল)
import os, time, subprocess
FAST_MODE = True  #@param {type:"boolean"}
USE_TUNNEL = True  #@param {type:"boolean"}
# Cloudflare টানেল শুধু বাইরে থেকে লিংকে ঢুকতে লাগে; বন্ধ করলে সার্ভার http://localhost:8000-এ চলবে
CF_BIN = globals().get("CF_BIN") or f"{LIB_DIR}/bin/cloudflared"
if USE_TUNNEL:
    for _ in range(60):  # cloudflared ইনস্টল সেলে ব্যাকগ্রাউন্ডে নামে
        if os.path.exists(CF_BIN) and os.access(CF_BIN, os.X_OK): break
        time.sleep(1)
    else:
        print("cloudflared এখনও নামেনি; টানেল ছাড়াই লোকাল সার্ভার চলবে")
        USE_TUNNEL = False

import threading, subprocess, re, secrets, tempfile, os, json, shutil, urllib.parse, soundfile as sf
import uvicorn
from fastapi import FastAPI, UploadFile, File, Form, HTTPException, Request
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response, FileResponse, JSONResponse, HTMLResponse
from pydantic import BaseModel

TOKEN = secrets.token_urlsafe(9)
cpu_lock = threading.Lock()
api = FastAPI(title="Any people bd voice clone")

@api.middleware("http")
async def _auth(request: Request, call_next):
    if request.url.path.startswith("/api") and request.method != "OPTIONS":
        if request.headers.get("x-token") != TOKEN:
            return JSONResponse({"detail": "টোকেন ভুল বা নেই"}, status_code=401)
    return await call_next(request)

api.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"], expose_headers=["X-Report"])

def _save_upload(f: UploadFile):
    ext = os.path.splitext(f.filename or "")[1] or ".wav"
    fd, tmp = tempfile.mkstemp(suffix=ext); os.close(fd)
    with open(tmp, "wb") as o: shutil.copyfileobj(f.file, o)
    return tmp

def _b(v): return str(v).lower() in ("1", "true", "yes", "on")

@api.get("/")
def index():
    p = next((q for q in (f"{LIB_DIR}/index_v2.html", f"{LIB_DIR}/index.html") if os.path.exists(q)), "")
    if p:
        html = open(p, encoding="utf-8").read().replace("Bangla Voice Lab", "Any people bd voice clone")
        return HTMLResponse(html)
    return HTMLResponse("<meta charset=utf-8><h3>index.html পাওয়া যায়নি</h3>Drive-এর BanglaVoiceLab ফোল্ডারে index.html রাখুন।")

@api.get("/api/health")
def health():
    return {"ok": True, "gpu": "CPU", "voices": len(list_voices())}

@api.get("/api/voices")
def voices():
    out = []
    for v in list_voices():
        m = json.load(open(f"{LIB_DIR}/voices/{v}/meta.json", encoding="utf-8"))
        out.append({"id": v, "name": m.get("name", v), "ref_text": m.get("ref_text", ""), "quality": m.get("quality", {})})
    return out

@api.post("/api/voices")
def voices_create(file: UploadFile = File(...), name: str = Form(...), ref_text: str = Form(""),
                  consent: str = Form("false"), denoise: str = Form("true"), pick_best: str = Form("true")):
    if not _b(consent): raise HTTPException(400, "সম্মতি (consent) দরকার")
    tmp = _save_upload(file)
    try:
        with cpu_lock:
            vid, meta = create_voice(name, tmp, ref_text, True, _b(denoise), _b(pick_best))
    except Exception as e:
        raise HTTPException(400, str(e))
    finally:
        os.remove(tmp)
    return {"id": vid, "name": meta["name"], "quality": meta["quality"], "ref_text": meta["ref_text"]}

@api.delete("/api/voices/{vid}")
def voices_delete(vid: str):
    delete_voice(vid); return {"ok": True}

@api.get("/api/voices/{vid}/ref")
def voices_ref(vid: str):
    p = f"{LIB_DIR}/voices/{vid}/ref.wav"
    if not os.path.exists(p): raise HTTPException(404, "নেই")
    return FileResponse(p, media_type="audio/wav")

@api.post("/api/transcribe")
def transcribe(file: UploadFile = File(...), denoise: str = Form("true"), pick_best: str = Form("true")):
    tmp = _save_upload(file)
    try:
        with cpu_lock:
            path, rep = prepare_reference(tmp, denoise=_b(denoise), pick_best=_b(pick_best), out_path=f"{WORK_DIR}/_tr.wav")
            text = transcribe_bn(path)
    except Exception as e:
        raise HTTPException(400, str(e))
    finally:
        os.remove(tmp)
    return {"text": text, "quality": rep["notes"]}

PROGRESS = {"pct": 0, "desc": "", "active": False}

@api.get("/api/progress")
def progress_ep():
    return PROGRESS

class TTSReq(BaseModel):
    voice: str
    text: str
    speed: float = 1.0
    steps: int = 16
    guidance: float = 2.0
    best_of: int = 1
    seed: int = -1
    normalize: bool = True
    fmt: str = "wav"
    lexicon: dict = {}

@api.post("/api/tts")
def tts(r: TTSReq):
    if not r.text.strip(): raise HTTPException(400, "টেক্সট খালি")
    if r.voice not in list_voices(): raise HTTPException(404, "ভয়েস পাওয়া যায়নি")
    normalizer.lex.update({k: v for k, v in r.lexicon.items() if k and v})
    steps_ = max(8, min(r.steps, 64)); best_ = max(1, min(r.best_of, 6))
    if FAST_MODE:  # index.html যা-ই পাঠাক, দ্রুত জেনারেশনের জন্য সীমা
        steps_, best_ = min(steps_, 16), 1
    def _prog(frac, desc=""):
        PROGRESS.update(pct=int(max(0.0, min(1.0, float(frac))) * 100), desc=desc, active=True)
    try:
        with cpu_lock:
            PROGRESS.update(pct=0, desc="শুরু", active=True)
            y, rep = synthesize_fast(r.text, r.voice, speed=r.speed, num_step=steps_,
                                guidance_scale=r.guidance, seed=r.seed, best_of=best_, do_normalize=r.normalize,
                                progress=_prog)
    except Exception as e:
        PROGRESS.update(active=False)
        raise HTTPException(500, str(e))
    PROGRESS.update(pct=100, desc="সম্পন্ন", active=False)
    fd, buf = tempfile.mkstemp(suffix=".wav"); os.close(fd); sf.write(buf, y, SR)
    mime = "audio/wav"
    if r.fmt == "mp3":
        mp3 = buf.replace(".wav", ".mp3")
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", buf, "-b:a", "192k", mp3], check=True)
        os.remove(buf); buf, mime = mp3, "audio/mpeg"
    data = open(buf, "rb").read(); os.remove(buf)
    return Response(data, media_type=mime, headers={"X-Report": urllib.parse.quote(rep)})

# ───────── রিয়েল-টাইম স্ট্রিমিং (Turbo Engine) ─────────
from fastapi.responses import StreamingResponse
import numpy as np

class TTSStreamReq(TTSReq):
    first_steps: int = 8
    batch_chars: int = 240

@api.post("/api/tts_stream")
def tts_stream(r: TTSStreamReq):
    if not r.text.strip(): raise HTTPException(400, "টেক্সট খালি")
    if r.voice not in list_voices(): raise HTTPException(404, "ভয়েস পাওয়া যায়নি")
    normalizer.lex.update({k: v for k, v in r.lexicon.items() if k and v})
    steps_ = max(4, min(r.steps, 64)); first_ = max(4, min(r.first_steps, 64))
    if FAST_MODE: steps_, first_ = min(steps_, 16), min(first_, 16)
    def gen():
        for a, pause in turbo_stream(r.text, r.voice, speed=r.speed, steps=steps_, first_steps=first_,
                                     guide=r.guidance, seed=r.seed, do_normalize=r.normalize,
                                     batch_chars=max(60, min(r.batch_chars, 600)), lock=cpu_lock):
            yield (np.clip(a, -1, 1) * 32767).astype("<i2").tobytes()
            yield np.zeros(int(SR * pause), dtype="<i2").tobytes()
    return StreamingResponse(gen(), media_type="application/octet-stream",
                             headers={"Cache-Control": "no-store", "X-Accel-Buffering": "no"})

@api.get("/api/turbo_stats")
def turbo_stats(): return dict(TURBO_LAST)

_LIVE_HTML = r"""<!doctype html><html lang="bn"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Any people bd voice clone — লাইভ</title>
<style>:root{--bg:#0f1420;--card:#171e2e;--line:#27314a;--tx:#e8ecf5;--mut:#8b97b3;--ac:#2dd4a0}
*{box-sizing:border-box}body{font:16px/1.6 "Noto Sans Bengali",system-ui,sans-serif;margin:0;background:radial-gradient(1000px 500px at 10% -10%,#1b2a4a55,transparent),var(--bg);color:var(--tx)}
.w{max-width:820px;margin:auto;padding:16px}h2{margin:6px 0 2px}.sub{color:var(--mut);font-size:14px;margin-bottom:12px}
.card{background:var(--card);border:1px solid var(--line);border-radius:14px;padding:14px;margin-bottom:12px}
label{display:block;color:var(--mut);font-size:13px;margin:8px 0 3px}
input,select,textarea{width:100%;padding:9px 11px;border-radius:10px;border:1px solid var(--line);background:var(--bg);color:inherit;font:inherit}
textarea{min-height:150px;resize:vertical}.g{display:grid;grid-template-columns:repeat(auto-fit,minmax(120px,1fr));gap:10px}
button{border:0;border-radius:10px;padding:10px 18px;font:inherit;font-weight:600;cursor:pointer;color:#04281d;background:linear-gradient(90deg,#2dd4a0,#38bdf8);margin:12px 8px 0 0}
button.s{background:#ff6b6b;color:#fff}button.d{background:transparent;border:1px solid var(--line);color:var(--tx)}button:disabled{opacity:.45;cursor:not-allowed}
canvas{width:100%;height:70px;display:block;border-radius:10px;background:var(--bg)}
.st{display:grid;grid-template-columns:repeat(auto-fit,minmax(120px,1fr));gap:8px;margin-top:10px}.st div{background:var(--bg);border:1px solid var(--line);border-radius:10px;padding:6px 10px;font-size:13px;color:var(--mut)}
.st b{display:block;color:var(--tx);font-size:17px}#m{color:var(--mut);margin-top:8px;white-space:pre-wrap;font-size:14px}</style></head><body><div class="w">
<h2>🎧 রিয়েল-টাইম ভয়েস</h2><div class="sub">Any people bd voice clone — প্রতিটি বাক্য তৈরি হওয়ামাত্র বাজে</div>
<div class="card"><canvas id="viz"></canvas>
<div class="st"><div>প্রথম শব্দ<b id="s1">—</b></div><div>বাজানো অডিও<b id="s2">—</b></div><div>এগিয়ে আছে (বাফার)<b id="s3">—</b></div><div>অবস্থা<b id="s4">প্রস্তুত</b></div></div></div>
<div class="card"><label>টোকেন</label><input id="tok" type="password">
<label>ভয়েস</label><select id="v"></select>
<label>টেক্সট</label><textarea id="t">আসসালামু আলাইকুম। আজ আবহাওয়া খুব সুন্দর। চলুন একসাথে ঘুরতে যাই। বিকেলে আমরা নদীর ধারে হাঁটব এবং চা খাব।</textarea>
<div class="g"><div><label>গতি</label><input id="sp" type="number" step="0.1" value="1.0"></div>
<div><label>স্টেপ (বাকি বাক্য)</label><input id="st" type="number" value="12"></div>
<div><label>প্রথম বাক্যের স্টেপ</label><input id="fs" type="number" value="8"></div>
<div><label>প্রি-বাফার (সেকেন্ড)</label><input id="pb" type="number" step="0.1" value="0"></div></div>
<button id="go">▶ বলো</button><button class="s" id="stop">■ থামাও</button><button class="d" id="dl" disabled>⬇ WAV সেভ</button><div id="m"></div></div></div>
<script>
const SR=24000,$=i=>document.getElementById(i);let ctx=null,an=null,ab=null,nextT=0,srcs=[],pcm=[],raf=0,t0=0;
const mt=location.hash.match(/token=([^&]+)/);if(mt)$("tok").value=decodeURIComponent(mt[1]);
const H=()=>({"x-token":$("tok").value.trim(),"Content-Type":"application/json"});
const msg=s=>$("m").textContent=s;
async function loadVoices(){try{const r=await fetch("/api/voices",{headers:H()});if(!r.ok)throw 0;
const a=await r.json();$("v").innerHTML=a.map(v=>`<option value="${v.id}">${v.name}</option>`).join("");
if(!a.length)msg("আগে একটি ভয়েস সেভ করুন")}catch(e){msg("ভয়েস লোড হয়নি (টোকেন ঠিক আছে?)")}}
function draw(){if(!an||!ctx)return;const d=new Uint8Array(an.frequencyBinCount);an.getByteFrequencyData(d);
const c=$("viz"),g=c.getContext("2d");c.width=c.clientWidth;c.height=70;g.clearRect(0,0,c.width,c.height);
const n=56,w=c.width/n;for(let i=0;i<n;i++){const v=d[i]/255,h=Math.max(2,v*c.height);g.fillStyle=`hsl(${150+i*3.2},85%,${45+v*20}%)`;g.fillRect(i*w+1,c.height-h,w-3,h)}
$("s3").textContent=Math.max(0,nextT-ctx.currentTime).toFixed(1)+" s";raf=requestAnimationFrame(draw)}
function stop(){if(ab)ab.abort();srcs.forEach(s=>{try{s.stop()}catch(e){}});srcs=[];cancelAnimationFrame(raf);if(ctx){ctx.close();ctx=null}}
function wavBlob(){const n=pcm.reduce((a,b)=>a+b.length,0),buf=new ArrayBuffer(44+n*2),v=new DataView(buf);
const ws=(o,s)=>{for(let i=0;i<s.length;i++)v.setUint8(o+i,s.charCodeAt(i))};
ws(0,"RIFF");v.setUint32(4,36+n*2,true);ws(8,"WAVEfmt ");v.setUint32(16,16,true);v.setUint16(20,1,true);v.setUint16(22,1,true);
v.setUint32(24,SR,true);v.setUint32(28,SR*2,true);v.setUint16(32,2,true);v.setUint16(34,16,true);ws(36,"data");v.setUint32(40,n*2,true);
let o=44;for(const p of pcm){for(let i=0;i<p.length;i++){v.setInt16(o,p[i],true);o+=2}}return new Blob([buf],{type:"audio/wav"})}
async function go(){stop();pcm=[];$("dl").disabled=true;$("s1").textContent="—";$("s2").textContent="0.0 s";
ctx=new(window.AudioContext||window.webkitAudioContext)({sampleRate:SR});await ctx.resume();
an=ctx.createAnalyser();an.fftSize=256;an.connect(ctx.destination);draw();
ab=new AbortController();nextT=ctx.currentTime+0.15+(+$("pb").value||0);t0=performance.now();let first=true,rest=new Uint8Array(0),tot=0;
$("s4").textContent="তৈরি হচ্ছে";msg("");
try{const r=await fetch("/api/tts_stream",{method:"POST",headers:H(),signal:ab.signal,body:JSON.stringify({voice:$("v").value,text:$("t").value,speed:+$("sp").value,steps:+$("st").value,first_steps:+$("fs").value})});
if(!r.ok){msg("ত্রুটি: "+(await r.text()));$("s4").textContent="ত্রুটি";return}
const rd=r.body.getReader();
while(true){const{done,value}=await rd.read();if(done)break;
const b=new Uint8Array(rest.length+value.length);b.set(rest);b.set(value,rest.length);
const n=b.length-(b.length%2);rest=b.slice(n);if(!n)continue;
const i16=new Int16Array(b.buffer.slice(0,n)),f=new Float32Array(i16.length);for(let i=0;i<f.length;i++)f[i]=i16[i]/32768;
pcm.push(i16);tot+=i16.length;
const buf=ctx.createBuffer(1,f.length,SR);buf.copyToChannel(f,0);const s=ctx.createBufferSource();s.buffer=buf;s.connect(an);
if(nextT<ctx.currentTime+0.05)nextT=ctx.currentTime+0.05;s.start(nextT);nextT+=buf.duration;srcs.push(s);
if(first){first=false;$("s1").textContent=Math.round(performance.now()-t0)+" ms";$("s4").textContent="বাজছে"}
$("s2").textContent=(tot/SR).toFixed(1)+" s"}
$("s4").textContent="শেষ ✅";$("dl").disabled=!pcm.length}catch(e){if(e.name!=="AbortError"){msg("ত্রুটি: "+e);$("s4").textContent="ত্রুটি"}}}
$("go").onclick=go;$("stop").onclick=()=>{stop();$("s4").textContent="থামানো";msg("")};
$("dl").onclick=()=>{const a=document.createElement("a");a.href=URL.createObjectURL(wavBlob());a.download="voice.wav";a.click()};
$("tok").onchange=loadVoices;loadVoices();
</script></body></html>"""

@api.get("/live")
def live_page():
    return HTMLResponse(_LIVE_HTML)

try:   # আগের সার্ভার থাকলে বন্ধ (পোর্ট দ্বন্দ্ব এড়াতে)
    srv.should_exit = True; time.sleep(1.5)
except NameError:
    pass
srv = uvicorn.Server(uvicorn.Config(api, host="0.0.0.0", port=8000, log_level="warning"))
threading.Thread(target=srv.run, daemon=True).start()

PUBLIC_URL = None
if USE_TUNNEL:
    try:
        tun = subprocess.Popen([CF_BIN, "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                               stderr=subprocess.PIPE, stdout=subprocess.DEVNULL, text=True)
        _found = threading.Event()
        def _read_tun():
            global PUBLIC_URL
            for line in tun.stderr:
                m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
                if m and not _found.is_set(): PUBLIC_URL = m.group(0); _found.set()
        threading.Thread(target=_read_tun, daemon=True).start()
        _found.wait(60)
    except Exception as e:
        print("Cloudflare টানেল চালু হয়নি:", e)
if not PUBLIC_URL:
    PUBLIC_URL = "http://localhost:8000"
    print("⚠️ পাবলিক URL পাওয়া যায়নি; লোকাল ঠিকানা দেখানো হচ্ছে")

print("=" * 60)
print("✅ লাইভ URL :", PUBLIC_URL)
print("🔑 টোকেন    :", TOKEN)
print("👉 এক ক্লিকে খুলুন:", f"{PUBLIC_URL}/#token={TOKEN}")
print("=" * 60)
print("🎧 রিয়েল-টাইম (বলার সাথে সাথে শোনা):", f"{PUBLIC_URL}/live#token={TOKEN}")
print("index.html Drive-এ না থাকলে আলাদাভাবে খুললে উপরের URL ও টোকেন UI-তে বসান।")

## ৮) Gradio ওয়েব UI (বাংলা)

In [ ]:
#@title UI চালু করুন
import gradio as gr, zipfile, tempfile, uuid, os

def ui_transcribe(audio, denoise, pick):
    if audio is None: return "", None, "অডিও দিন"
    path, rep = prepare_reference(audio, denoise=denoise, pick_best=pick, out_path=f"{WORK_DIR}/ref_preview.wav")
    return transcribe_bn(path), path, "\n".join(rep["notes"]) + f"\nদৈর্ঘ্য: {rep['duration_sec']}s | SNR: {rep['snr_db']} dB"

def ui_save(name, audio, text, consent, denoise, pick):
    if not name or audio is None: return "নাম ও অডিও দিন", gr.update()
    try:
        vid, meta = create_voice(name, audio, text, consent, denoise, pick)
    except Exception as e:
        return f"❌ {e}", gr.update()
    return f"✅ সেভ হয়েছে: {vid}\n" + "\n".join(meta["quality"]["notes"]), gr.update(choices=list_voices(), value=vid)

def ui_generate(vid, text, speed, steps, guide, best_of, seed, norm, progress=gr.Progress()):
    if not vid: return None, "আগে একটি ভয়েস বেছে নিন"
    y, rep = synthesize_fast(text, vid, speed=speed, num_step=int(steps), guidance_scale=guide,
                             seed=int(seed), best_of=int(best_of), do_normalize=norm, progress=progress)
    path = f"{LIB_DIR}/outputs/{uuid.uuid4().hex[:8]}.wav"
    sf.write(path, y, SR)
    return path, rep

def ui_batch(vid, lines, speed, steps, best_of, norm, progress=gr.Progress()):
    rows = [l for l in lines.splitlines() if l.strip()]
    if not vid or not rows: return None
    fd, zpath = tempfile.mkstemp(suffix=".zip"); os.close(fd)
    with zipfile.ZipFile(zpath, "w") as z:
        for i, line in enumerate(rows, 1):
            progress((i - 1) / len(rows), desc=f"{int((i - 1) / len(rows) * 100)}% — লাইন {i}/{len(rows)}")
            y, _ = synthesize_fast(line, vid, speed=speed, num_step=int(steps), best_of=int(best_of), do_normalize=norm)
            p = f"{WORK_DIR}/_b{i:03d}.wav"; sf.write(p, y, SR); z.write(p, f"{i:03d}.wav"); os.remove(p)
    progress(1.0, desc="১০০% — সম্পন্ন")
    return zpath

def ui_design(text, gender, age, pitch, whisper):
    inst = ", ".join(x for x in [gender, age, pitch, "whisper" if whisper else ""] if x and x != "—")
    y, rep = synthesize(text, instruct=inst or None, best_of=1)
    return (SR, y), "instruct: " + inst + "\n(Voice Design মূলত চীনা/ইংরেজিতে ট্রেইন্ড; বাংলায় অস্থির হতে পারে। ক্লোনিংই সবচেয়ে স্থিতিশীল।)"

with gr.Blocks(title="Any people bd voice clone", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🇧🇩 Any people bd voice clone\nশুধু নিজের বা অনুমতিপ্রাপ্ত কণ্ঠ ক্লোন করুন।")
    with gr.Tab("১. ভয়েস তৈরি"):
        with gr.Row():
            with gr.Column():
                v_name = gr.Textbox(label="ভয়েসের নাম")
                v_audio = gr.Audio(label="রেফারেন্স অডিও (৫–১৫ সেকেন্ড, পরিষ্কার)", type="filepath", sources=["upload", "microphone"])
                v_den = gr.Checkbox(True, label="নয়েজ রিমুভ"); v_pick = gr.Checkbox(True, label="সেরা অংশ স্বয়ংক্রিয় বাছাই")
                b_tr = gr.Button("ক্লিন + ট্রান্সক্রাইব")
            with gr.Column():
                v_prev = gr.Audio(label="প্রসেসড রেফারেন্স", interactive=False)
                v_text = gr.Textbox(label="রেফারেন্সের ট্রান্সক্রিপ্ট (ভুল থাকলে ঠিক করুন, এটি উচ্চারণের মান বাড়ায়)", lines=3)
                v_rep = gr.Textbox(label="কোয়ালিটি রিপোর্ট", lines=3)
                v_cons = gr.Checkbox(False, label="আমি নিশ্চিত করছি: এটি আমার নিজের কণ্ঠ বা কণ্ঠধারীর লিখিত/স্পষ্ট অনুমতি আছে")
                b_save = gr.Button("ভয়েস সেভ করুন", variant="primary"); v_status = gr.Textbox(label="স্ট্যাটাস")
    with gr.Tab("২. ভয়েস জেনারেট"):
        with gr.Row():
            with gr.Column():
                g_voice = gr.Dropdown(list_voices(), label="ভয়েস", value=(list_voices() or [None])[0])
                g_text = gr.Textbox(label="বাংলা টেক্সট", lines=7, value="আজ আবহাওয়া খুব সুন্দর। চলো, একটু হেঁটে আসি।")
                with gr.Accordion("উন্নত সেটিংস", open=False):
                    g_speed = gr.Slider(0.7, 1.4, 1.0, step=0.05, label="গতি")
                    g_steps = gr.Slider(8, 64, 16, step=4, label="কোয়ালিটি স্টেপ (বেশি = ভালো, ধীর)")
                    g_guide = gr.Slider(1.0, 4.0, 2.0, step=0.1, label="Guidance scale")
                    g_best = gr.Slider(1, 6, 1, step=1, label="Best-of-N (ASR দিয়ে সেরা উচ্চারণ বাছাই)")
                    g_seed = gr.Number(-1, label="Seed (-1 = এলোমেলো)")
                    g_norm = gr.Checkbox(True, label="বাংলা নরমালাইজেশন (সংখ্যা/তারিখ/টাকা)")
                b_gen = gr.Button("🔊 জেনারেট", variant="primary")
            with gr.Column():
                g_out = gr.Audio(label="ফলাফল", type="filepath"); g_rep = gr.Textbox(label="রিপোর্ট", lines=8)
        gr.Button("🔄 ভয়েস তালিকা রিফ্রেশ").click(lambda: gr.update(choices=list_voices()), None, g_voice)
    with gr.Tab("৩. ব্যাচ"):
        b_voice = gr.Dropdown(list_voices(), label="ভয়েস"); b_lines = gr.Textbox(label="প্রতি লাইনে একটি বাক্য/অনুচ্ছেদ", lines=8)
        with gr.Row():
            b_speed = gr.Slider(0.7, 1.4, 1.0, step=0.05, label="গতি"); b_steps = gr.Slider(8, 64, 16, step=4, label="স্টেপ")
            b_best = gr.Slider(1, 4, 1, step=1, label="Best-of-N"); b_norm = gr.Checkbox(True, label="নরমালাইজ")
        b_run = gr.Button("ব্যাচ জেনারেট (ZIP)", variant="primary"); b_zip = gr.File(label="ZIP")
    with gr.Tab("৪. ভয়েস ডিজাইন (পরীক্ষামূলক)"):
        d_text = gr.Textbox(label="বাংলা টেক্সট", value="এটি একটি ভয়েস ডিজাইন পরীক্ষা।")
        with gr.Row():
            d_g = gr.Dropdown(["—", "male", "female"], value="female", label="লিঙ্গ")
            d_a = gr.Dropdown(["—", "child", "young adult", "middle-aged", "elderly"], value="young adult", label="বয়স")
            d_p = gr.Dropdown(["—", "very low pitch", "low pitch", "moderate pitch", "high pitch", "very high pitch"], value="moderate pitch", label="পিচ")
            d_w = gr.Checkbox(False, label="ফিসফিস")
        d_b = gr.Button("জেনারেট"); d_out = gr.Audio(label="ফলাফল"); d_rep = gr.Textbox(label="নোট")

    b_tr.click(ui_transcribe, [v_audio, v_den, v_pick], [v_text, v_prev, v_rep])
    b_save.click(ui_save, [v_name, v_audio, v_text, v_cons, v_den, v_pick], [v_status, g_voice])
    b_gen.click(ui_generate, [g_voice, g_text, g_speed, g_steps, g_guide, g_best, g_seed, g_norm], [g_out, g_rep])
    b_run.click(ui_batch, [b_voice, b_lines, b_speed, b_steps, b_best, b_norm], b_zip)
    d_b.click(ui_design, [d_text, d_g, d_a, d_p, d_w], [d_out, d_rep])

demo.queue().launch(share=True, debug=False)

---
## ৯) 🚀 অ্যাডভান্সড: LoRA ফাইন-টিউনিং (নিজের স্পিকারের আলাদা মডেল)

**কখন দরকার?** জিরো-শট ক্লোনিং ভালো কিন্তু উচ্চারণ/টোন/ক্যারেক্টার আরও নিখুঁত চাইলে।
**কী লাগবে:** একই ব্যক্তির ১০–৬০ মিনিট পরিষ্কার বাংলা রেকর্ডিং (একটি ফোল্ডারে wav/mp3/flac)। T4-তে ২০০০ স্টেপ সাধারণত ১–২ ঘণ্টা।

**ধাপ:** ডেটা কেটে ট্রান্সক্রাইব → (ঐচ্ছিক) ট্রান্সক্রিপ্ট ঠিক করুন → টোকেনাইজ → LoRA ট্রেন → অ্যাডাপ্টার লোড।

In [ ]:
#@title ৯.১ ডেটাসেট তৈরি (কাটা + বাংলা ট্রান্সক্রিপশন)
# disabled on CPU (LoRA fine-tuning needs GPU)
# import glob, json, os, random, numpy as np, soundfile as sf, librosa
# from tqdm.auto import tqdm

# RAW_DIR = "/content/drive/MyDrive/my_recordings"  #@param {type:"string"}
# FT_DIR = f"{LIB_DIR}/finetune"
# CLIP_DIR = f"{FT_DIR}/clips"
# os.makedirs(CLIP_DIR, exist_ok=True)

# def make_clips(y, sr=SR, min_s=3.0, max_s=14.0, top_db=32, max_gap=0.5):
#     segs = librosa.effects.split(y, top_db=top_db, frame_length=1024, hop_length=256)
#     clips, cur = [], None
#     for s, e in segs:
#         if cur is None: cur = [s, e]; continue
#         if (s - cur[1]) / sr <= max_gap and (e - cur[0]) / sr <= max_s:
#             cur[1] = e
#         else:
#             clips.append(tuple(cur)); cur = [s, e]
#     if cur: clips.append(tuple(cur))
#     pad = int(0.08 * sr)
#     return [y[max(0, s - pad): e + pad] for s, e in clips if min_s <= (e - s) / sr <= max_s + 1]

# files_ = sorted(sum([glob.glob(f"{RAW_DIR}/**/*.{ext}", recursive=True) for ext in ("wav", "mp3", "flac", "m4a", "ogg")], []))
# print(len(files_), "ফাইল পাওয়া গেছে")
# rows, n = [], 0
# for f in tqdm(files_):
#     y = enhance(load_mono(f), denoise=True)
#     for c in make_clips(y):
#         n += 1; p = f"{CLIP_DIR}/c{n:05d}.wav"; sf.write(p, c, SR)
#         txt = normalizer.normalize(transcribe_array(c))
#         if len(_clean_for_cer(txt)) >= 5:
#             rows.append({"id": f"c{n:05d}", "audio_path": p, "text": txt, "language_id": "bn"})
# with open(f"{FT_DIR}/manifest_raw.jsonl", "w", encoding="utf-8") as fo:
#     for r in rows: fo.write(json.dumps(r, ensure_ascii=False) + "\n")
# total = sum(sf.info(r["audio_path"]).duration for r in rows) / 60
# print(f"{len(rows)} ক্লিপ, মোট {total:.1f} মিনিট → {FT_DIR}/manifest_raw.jsonl")
# print("💡 সময় থাকলে manifest_raw.jsonl খুলে ভুল ট্রান্সক্রিপ্ট ঠিক করুন, এতে মান অনেক বাড়ে।")

In [ ]:
#@title ৯.২ টোকেনাইজ ও ট্রেন কনফিগ
# disabled on CPU (LoRA fine-tuning needs GPU)
# import json, random, os, subprocess

# STEPS = 2000  #@param {type:"integer"}
# LEARNING_RATE = 1e-4  #@param {type:"number"}

# rows = [json.loads(l) for l in open(f"{FT_DIR}/manifest_raw.jsonl", encoding="utf-8")]
# random.Random(42).shuffle(rows)
# nd = max(5, int(0.05 * len(rows)))
# for name, part in (("dev", rows[:nd]), ("train", rows[nd:])):
#     with open(f"{FT_DIR}/{name}.jsonl", "w", encoding="utf-8") as fo:
#         for r in part: fo.write(json.dumps(r, ensure_ascii=False) + "\n")

# env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
# for split in ("train", "dev"):
#     subprocess.run(["python", "-m", "omnivoice.scripts.extract_audio_tokens",
#         "--input_jsonl", f"{FT_DIR}/{split}.jsonl",
#         "--tar_output_pattern", f"{FT_DIR}/tokens/{split}/audios/shard-%06d.tar",
#         "--jsonl_output_pattern", f"{FT_DIR}/tokens/{split}/txts/shard-%06d.jsonl",
#         "--tokenizer_path", "eustlb/higgs-audio-v2-tokenizer", "--nj_per_gpu", "1", "--shuffle", "True"],
#         check=True, env=env)

# bf16_ok = False
# train_cfg = {
#   "llm_name_or_path": "Qwen/Qwen3-0.6B", "audio_vocab_size": 1025, "audio_mask_id": 1024, "num_audio_codebook": 8,
#   "audio_codebook_weights": [8, 8, 6, 6, 4, 4, 2, 2], "drop_cond_ratio": 0.1,
#   "prompt_ratio_range": [0.0, 0.3], "mask_ratio_range": [0.0, 1.0], "language_ratio": 0.8,
#   "use_pinyin_ratio": 0.0, "instruct_ratio": 0.0, "only_instruct_ratio": 0.0,
#   "resume_from_checkpoint": None, "init_from_checkpoint": MODEL_ID,
#   "use_lora": True, "lora_r": 32, "lora_alpha": 64, "lora_dropout": 0.05, "lora_bias": "none",
#   "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
#   "lora_modules_to_save": ["audio_embeddings", "audio_heads"],
#   "learning_rate": LEARNING_RATE, "weight_decay": 0.01, "max_grad_norm": 1.0, "steps": STEPS, "seed": 42,
#   "warmup_type": "ratio", "warmup_ratio": 0.05, "warmup_steps": 0,
#   "attn_implementation": "sdpa", "max_sample_tokens": 2000, "min_sample_tokens": 50, "max_batch_size": 32,
#   "batch_tokens": 4096, "gradient_accumulation_steps": 2, "num_workers": 2,
#   "mixed_precision": "no", "allow_tf32": True,
#   "logging_steps": 25, "eval_steps": 250, "save_steps": 250, "keep_last_n_checkpoints": 3,
# }
# data_cfg = {"train": [{"manifest_path": [f"{FT_DIR}/tokens/train/data.lst"]}],
#             "dev": [{"manifest_path": [f"{FT_DIR}/tokens/dev/data.lst"]}]}
# json.dump(train_cfg, open(f"{FT_DIR}/train_config.json", "w"), indent=2)
# json.dump(data_cfg, open(f"{FT_DIR}/data_config.json", "w"), indent=2)
# print("টোকেনাইজ সম্পন্ন, কনফিগ তৈরি। mixed_precision =", train_cfg["mixed_precision"])

In [ ]:
#@title ৯.৩ LoRA ট্রেনিং শুরু (চেকপয়েন্ট Drive-এ সেভ হবে)
# disabled on CPU (LoRA fine-tuning needs GPU)
# OUT_DIR = f"{FT_DIR}/lora_out"
# !accelerate launch --num_processes 1 -m omnivoice.cli.train \
#     --train_config {FT_DIR}/train_config.json --data_config {FT_DIR}/data_config.json --output_dir {OUT_DIR}
# !ls {OUT_DIR}

In [ ]:
#@title ৯.৪ ফাইন-টিউন করা অ্যাডাপ্টার লোড ও পরীক্ষা
# disabled on CPU (LoRA fine-tuning needs GPU)
# import glob, os
# from omnivoice.utils.lora import load_lora_adapter
# ckpts = sorted(glob.glob(f"{FT_DIR}/lora_out/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
# ADAPTER = ckpts[-1]  # চাইলে অন্য চেকপয়েন্ট বেছে নিন
# print("অ্যাডাপ্টার:", ADAPTER)
# model = load_lora_adapter(model, ADAPTER)   # মেমোরিতে মার্জ হয়, বাকি কোড অপরিবর্তিত
# model.load_asr_model()                      # ASR আবার লোড (best-of-N এর জন্য)
# _cache.clear()
# y, rep = synthesize("এটি আমার ফাইন-টিউন করা কণ্ঠের পরীক্ষা। কেমন শোনাচ্ছে?", voice_id=list_voices()[0], best_of=3)
# print(rep); display(Audio(y, rate=SR))
# # স্থায়ী ব্যবহারের জন্য মার্জ: !omnivoice-merge-lora --base_model k2-fsa/OmniVoice --lora_adapter {ADAPTER} --output_dir {FT_DIR}/merged
# # তারপর উপরের MODEL_ID = "{FT_DIR}/merged" দিয়ে মডেল লোড করুন।

## 💡 সেরা ফল পাওয়ার টিপস
- **রেফারেন্স:** একই ভাষায় (বাংলা), ৫–১০ সেকেন্ড, একজন কথা বলছে, ব্যাকগ্রাউন্ড মিউজিক/নয়েজ নেই, স্বাভাবিক গতিতে পড়া।
- **ট্রান্সক্রিপ্ট ঠিক করুন:** Whisper-এর ভুল রেফারেন্স টেক্সট উচ্চারণ নষ্ট করে। ইউআই-তে ঠিক করে সেভ করুন।
- **Best-of-N:** ডিফল্ট ১ (সবচেয়ে দ্রুত)। উচ্চারণ নিখুঁত চাইলে ৩–৪ দিন, তবে সময় বাড়বে।
- **সংখ্যা/ইংরেজি শব্দ:** `CUSTOM_LEXICON`-এ নিজের শব্দ যোগ করুন (যেমন `"Wi-Fi": "ওয়াই-ফাই"`)।
- **ইংরেজি ও বাংলা মিশ্রণ:** রেফারেন্স বাংলা থাকলে ইংরেজি শব্দে বাংলা টান আসতে পারে (cross-lingual নিয়ম)।
- **ফাইন-টিউনিং:** ৩০+ মিনিট ডেটা ও ঠিক করা ট্রান্সক্রিপ্ট হলে সবচেয়ে বড় উন্নতি।
- নোটবুক মাঝপথে বন্ধ হলে Drive থেকে ভয়েস প্রোফাইল ফিরে পাবেন (`BanglaVoiceLab/voices`)।